In [31]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor
)
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
import tensorflow as tf
from sklearn.model_selection import GridSearchCV
import joblib


In [2]:
raw_df = pd.read_csv("./SolarPanelData.csv").drop(columns=['Unnamed: 0'])
df = raw_df.copy()
df.head()
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values("Date")

In [3]:
def get_season(date):
    m = date.month
    if m in [3, 4, 5]:
        return "Spring"
    elif m in [6, 7, 8]:
        return "Summer"
    elif m in [9, 10, 11]:
        return "Fall"
    else:
        return "Winter"

In [4]:
df['Season'] = df['Date'].apply(get_season).astype(str)

df['Month'] = df['Date'].dt.month
df['DayOfYear'] = df['Date'].dt.dayofyear

df['PressureBin'] = pd.qcut(df['Pressure'], q=5, duplicates='drop').astype(str)

In [5]:
X = df.drop(columns=['Date', 'Solar(PV)', 'Pressure'])
y = df['Solar(PV)']

In [6]:
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.1, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.1, random_state=42)

In [7]:
X_train['Solar_lag1'] = y_train.shift(1)
X_train['Solar_lag7'] = y_train.shift(7)

X_val['Solar_lag1'] = y_val.shift(1)
X_val['Solar_lag7'] = y_val.shift(7)

X_test['Solar_lag1'] = y_test.shift(1)
X_test['Solar_lag7'] = y_test.shift(7)

In [8]:
train_mask = X_train[['Solar_lag1','Solar_lag7']].notna().all(axis=1)

X_train = X_train[train_mask]
y_train = y_train[train_mask]

val_mask = X_val[['Solar_lag1','Solar_lag7']].notna().all(axis=1)

X_val = X_val[val_mask]
y_val = y_val[val_mask]

test_mask = X_test[['Solar_lag1','Solar_lag7']].notna().all(axis=1)

X_test = X_test[test_mask]
y_test = y_test[test_mask]

In [9]:
display(X_train.columns)
num_cols = [
'AvgTemperture', 'AverageDew(point via humidity)', 'Humidity', 'Wind',
'Month', 'DayOfYear', 'Solar_lag1', 'Solar_lag7'
]
cat_cols = ['Season', 'PressureBin']

Index(['AvgTemperture', 'AverageDew(point via humidity)', 'Humidity', 'Wind',
       'Season', 'Month', 'DayOfYear', 'PressureBin', 'Solar_lag1',
       'Solar_lag7'],
      dtype='object')

In [10]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", StandardScaler(), num_cols)
    ]
)

In [33]:
results = []

models = {
    "DecisionTree": DecisionTreeRegressor(random_state=42),
    "RandomForest": RandomForestRegressor(random_state=42),
    "ExtraTrees": ExtraTreesRegressor(random_state=42),
    "GradientBoosting": GradientBoostingRegressor(random_state=42),
    "XGBoost": XGBRegressor(random_state=42)
}

for name, model in models.items():
    pipe = Pipeline([
        ("preprocess", preprocessor),
        ("model", model)
    ])

    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_val)

    results.append({
        "Model": name,
        "R2": r2_score(y_val, y_pred),
        "MAE": mean_absolute_error(y_val, y_pred),
        "RMSE": root_mean_squared_error(y_val, y_pred)
    })

results_df = pd.DataFrame(results).sort_values("RMSE")
results_df


,Model,R2,MAE,RMSE
4,XGBoost,0.873338,2.401595,2.776464
2,ExtraTrees,0.842164,2.486358,3.099351
1,RandomForest,0.827568,2.845353,3.239490
3,GradientBoosting,0.803804,2.787926,3.455518
0,DecisionTree,0.679604,3.298857,4.415823


In [ ]:
xgb = models['XGBoost']

pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", xgb)
])

pipeline.fit(X_train, y_train)

ohe = pipeline.named_steps["preprocess"].named_transformers_["cat"]

ohe_features = list(ohe.get_feature_names_out(cat_cols))

all_features = ohe_features + num_cols

xgb = pipeline.named_steps["model"]
importances = xgb.get_booster().get_score(importance_type='gain')

rows = []
for f, imp in importances.items():
    idX = int(f[1:])
    rows.append((all_features[idX], imp))

fi = (
    pd.DataFrame(rows, columns=['Feature', 'Importance'])
    .sort_values(by='Importance', ascending=False)
)

display(fi)

y_pred = pipeline.predict(X_val)

print("R²:", r2_score(y_val, y_pred))
print("MAE:", mean_absolute_error(y_val, y_pred))
print("RMSE:", root_mean_squared_error(y_val, y_pred))

,Feature,Importance
2,Season_Summer,54.016632
0,Season_Fall,48.046082
8,AvgTemperture,27.255764
9,AverageDew(point via humidity),7.955474
4,"PressureBin_(28.898999999999997, 29.1]",7.116208
10,Humidity,3.574533
7,"PressureBin_(29.3, 29.6]",2.863962
11,Wind,1.385124
12,Month,1.156673
1,Season_Spring,0.993100


R²: 0.8733375485386466
MAE: 2.4015952452795832
RMSE: 2.776464009490229


In [12]:
class FFRegressor:
    def __init__(self, num_cols, cat_cols):
        self.num_cols = num_cols
        self.cat_cols = cat_cols

        self.inputs = {}
        self.encoded = []

    def build_preprocessing(self, X_train):
        """
        Builds preprocessing layers and adapts them on training data.
        """

        for col in self.num_cols:
            inp = tf.keras.Input(shape=(1,), name=col, dtype="float32")
            self.inputs[col] = inp

            normalizer = tf.keras.layers.Normalization(axis=None)
            normalizer.adapt(X_train[col].to_numpy().reshape(-1, 1))

            self.encoded.append(normalizer(inp))

        for col in self.cat_cols:
            inp = tf.keras.Input(shape=(1,), name=col, dtype="string")
            self.inputs[col] = inp

            lookup = tf.keras.layers.StringLookup(output_mode="binary")
            lookup.adapt(X_train[col].to_numpy().reshape(-1, 1))

            self.encoded.append(lookup(inp))

    def build_model(self):
        """
        Builds and returns the compiled Keras model.
        """

        all_features = tf.keras.layers.Concatenate()(self.encoded)

        X = tf.keras.layers.Dense(24, activation="relu")(all_features)
        outputs = tf.keras.layers.Dense(1, activation="relu")(X)

        model = tf.keras.Model(inputs=self.inputs, outputs=outputs)

        model.compile(
            optimizer="sgd",
            loss="mse",
            metrics=[tf.keras.metrics.RootMeanSquaredError()]
        )

        return model


In [13]:
X_train_ff = X_train.copy()
X_val_ff   = X_val.copy()

X_train_ff_dict = {
    col: X_train_ff[col].to_numpy()
    for col in num_cols + cat_cols
}

X_val_ff_dict = {
    col: X_val_ff[col].to_numpy()
    for col in num_cols + cat_cols
}


In [14]:
regressor = FFRegressor(num_cols, cat_cols)
regressor.build_preprocessing(X_train)
model = regressor.build_model()

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    "models/best_ffr.keras",
    monitor="val_loss",
    save_best_only=True
)


history = model.fit(
    x=X_train_ff_dict,
    y=y_train,
    epochs=50,
    batch_size=3,
    validation_data=(X_val_ff_dict, y_val),
    callbacks=[checkpoint]
)

Epoch 1/50
105/105 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 103.6512 - root_mean_squared_error: 10.1809 - val_loss: 50.9136 - val_root_mean_squared_error: 7.1354
Epoch 2/50
105/105 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 66.7553 - root_mean_squared_error: 8.1704 - val_loss: 44.2663 - val_root_mean_squared_error: 6.6533
Epoch 3/50
105/105 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 60.0143 - root_mean_squared_error: 7.7469 - val_loss: 36.2552 - val_root_mean_squared_error: 6.0212
Epoch 4/50
105/105 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 56.6331 - root_mean_squared_error: 7.5255 - val_loss: 35.1888 - val_root_mean_squared_error: 5.9320
Epoch 5/50
105/105 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 42.2681 - root_mean_squared_error: 6.5014 - val_loss: 26.6935 - val_root_mean_squared_error: 5.1666
Epoch 6/50
105/105 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 36.3881 - root_mean_squared_error: 6.0323 - val_loss: 35.3321 - val_root_mean_squared_error: 5.9441
Epoch 7/50
105/105 ━━━━━━━━━━━━━━━━━━━

In [15]:
def make_sequence_1d(x, timesteps):
    x = x.to_numpy()
    return np.array([
        x[i:i+timesteps]
        for i in range(len(x) - timesteps)
    ])

In [16]:
X_train_seq_dict = {}
X_val_seq_dict = {}

for col in num_cols:
    seq = make_sequence_1d(X_train[col], 7).astype("float32")
    X_train_seq_dict[col] = tf.convert_to_tensor(seq[:, :, None], dtype=tf.float32)

    seq_val = make_sequence_1d(X_val[col], 7).astype("float32")
    X_val_seq_dict[col] = tf.convert_to_tensor(seq_val[:, :, None], dtype=tf.float32)

for col in cat_cols:
    seq = make_sequence_1d(X_train[col].astype(str), 7)
    
    X_train_seq_dict[col] = tf.convert_to_tensor(seq[:, :, None], dtype=tf.string)

    seq_val = make_sequence_1d(X_val[col].astype(str), 7)
    X_val_seq_dict[col] = tf.convert_to_tensor(seq_val[:, :, None], dtype=tf.string)

y_train_seq = tf.convert_to_tensor(y_train.to_numpy()[7:].astype("float32"))
y_val_seq = tf.convert_to_tensor(y_val.to_numpy()[7:].astype("float32"))

In [17]:

print(X_train_seq_dict['AvgTemperture'][0])
print(X_train_seq_dict['AvgTemperture'].shape)

tf.Tensor(
[[76.4]
 [65.5]
 [66.5]
 [80.6]
 [77.7]
 [97.5]
 [75.1]], shape=(7, 1), dtype=float32)
(308, 7, 1)


In [18]:
class RNNRegressor:
    def __init__(self, num_cols, cat_cols, timesteps=7):
        self.num_cols = num_cols
        self.cat_cols = cat_cols
        self.timesteps = timesteps

        self.num_normalizers = {}
        self.cat_encoders = {}

    def adapt_preprocessing(self, X_train):

        for col in self.num_cols:
            normalizer = tf.keras.layers.Normalization(axis=None)
            normalizer.adapt(X_train[col].to_numpy().reshape(-1,1))
            self.num_normalizers[col] = normalizer

        for col in self.cat_cols:
            lookup = tf.keras.layers.StringLookup(output_mode='binary')
            lookup.adapt(X_train[col].to_numpy().reshape(-1,1))
            self.cat_encoders[col] = lookup

    def build_model(self):
        inputs = {}
        encoded_steps = []

        for col in self.num_cols:
            inp = tf.keras.layers.Input((self.timesteps, 1), name=col, dtype='float32')
            inputs[col] = inp

            x = tf.keras.layers.TimeDistributed(self.num_normalizers[col])(inp)
            encoded_steps.append(x)

        for col in self.cat_cols:
            inp = tf.keras.layers.Input((self.timesteps, 1), name=col, dtype='string')
            inputs[col] = inp

            x = tf.keras.layers.TimeDistributed(self.cat_encoders[col])(inp)
            encoded_steps.append(x)

        x = tf.keras.layers.Concatenate(axis=-1)(encoded_steps)

        x = tf.keras.layers.LSTM(32, activation='tanh')(x)
        x = tf.keras.layers.Dropout(0.5)(x)
        outputs = tf.keras.layers.Dense(1, activation='relu')(x)

        model = tf.keras.Model(inputs=inputs, outputs=outputs)

        model.compile(
            optimizer='adamW',
            loss="mse",
            metrics=['root_mean_squared_error']
        )

        return model

In [19]:
rnn = RNNRegressor(num_cols, cat_cols, timesteps=7)

rnn.adapt_preprocessing(X_train)
model = rnn.build_model()

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    "models/best_rnnr.keras",
    monitor="val_loss",
    save_best_only=True
)

history = model.fit(
    x=X_train_seq_dict,
    y=y_train_seq,
    epochs=50,
    batch_size=3,
    validation_data=(X_val_seq_dict, y_val_seq),
    callbacks=[checkpoint]
)


Epoch 1/50
103/103 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - loss: 537.7645 - root_mean_squared_error: 23.1897 - val_loss: 323.1336 - val_root_mean_squared_error: 17.9759
Epoch 2/50
103/103 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 242.4734 - root_mean_squared_error: 15.5716 - val_loss: 203.6218 - val_root_mean_squared_error: 14.2696
Epoch 3/50
103/103 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 164.0760 - root_mean_squared_error: 12.8092 - val_loss: 149.3327 - val_root_mean_squared_error: 12.2202
Epoch 4/50
103/103 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 127.3249 - root_mean_squared_error: 11.2838 - val_loss: 118.2303 - val_root_mean_squared_error: 10.8734
Epoch 5/50
103/103 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 105.7391 - root_mean_squared_error: 10.2830 - val_loss: 97.4128 - val_root_mean_squared_error: 9.8698
Epoch 6/50
103/103 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 92.2506 - root_mean_squared_error: 9.6047 - val_loss: 86.2233 - val_root_mean_squared_error: 9.2857
Epoch 7/50
103/103 ━━━

In [20]:
new_num_cols = ['DayOfYear', 'AvgTemperture', 'Wind', 'Solar_lag1', 'Solar_lag7']
new_cat_cols=['Season', 'PressureBin']

In [21]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), new_cat_cols),
        ("num", StandardScaler(), new_num_cols)
    ]
)

model = XGBRegressor(random_state=42)

pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", model)
])

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_val)

print("R²:", r2_score(y_val, y_pred))
print("MAE:", mean_absolute_error(y_val, y_pred))
print("RMSE:", root_mean_squared_error(y_val, y_pred))

R²: 0.8733375485386466
MAE: 2.4015952452795832
RMSE: 2.776464009490229


In [22]:
param_grid = {
    "model__n_estimators": [100, 300, 500, 800],
    "model__learning_rate": [0.01, 0.05, 0.1],
    "model__max_depth": [2, 3, 4, 5, None],
    "model__min_child_weight": [1, 3, 5, 10],
    "model__subsample": [0.6, 0.8, 1.0],
    "model__colsample_bytree": [0.6, 0.8, 1.0],
    "model__gamma": [0, 0.1, 0.3, 1.0],
}

grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",
    cv=5,               
    n_jobs=-1,         
    verbose=2
)

grid.fit(X_train, y_train)

print("Best CV RMSE:", -grid.best_score_)
print("Best params:")
for k, v in grid.best_params_.items():
    print(f"  {k}: {v}")

best_xgb_model = grid.best_estimator_

y_pred_xgb = best_xgb_model.predict(X_val)

print("Validation R²:", r2_score(y_val, y_pred_xgb))
print("Validation MAE:", mean_absolute_error(y_val, y_pred_xgb))
print("Validation RMSE:", root_mean_squared_error(y_val, y_pred_xgb))


Fitting 5 folds for each of 8640 candidates, totalling 43200 fits
Best CV RMSE: 3.838254716191686
Best params:
  model__colsample_bytree: 0.8
  model__gamma: 1.0
  model__learning_rate: 0.01
  model__max_depth: None
  model__min_child_weight: 1
  model__n_estimators: 800
  model__subsample: 0.8
Validation R²: 0.858971641571572
Validation MAE: 2.5009766742425747
Validation RMSE: 2.929687708354021


In [23]:
et = ExtraTreesRegressor(
    random_state=42,
    n_jobs=-1
)

param_grid_et = {
    "model__n_estimators": [200, 500, 800],
    "model__max_depth": [None, 5, 10, 20],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 5],
    "model__max_features": ["sqrt", "log2", 0.5, 1.0],
    "model__bootstrap": [False, True],
}

pipeline_et = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", et)
])

grid_et = GridSearchCV(
    estimator=pipeline_et,
    param_grid=param_grid_et,
    scoring="neg_root_mean_squared_error",
    cv=5,
    n_jobs=-1,
    verbose=2
)

grid_et.fit(X_train, y_train)

print("Best CV RMSE (ExtraTrees):", -grid_et.best_score_)
print("Best params (ExtraTrees):")
for k, v in grid_et.best_params_.items():
    print(f"  {k}: {v}")

best_et_model = grid_et.best_estimator_
y_pred_et = best_et_model.predict(X_val)

print("Validation R²:", r2_score(y_val, y_pred_et))
print("Validation MAE:", mean_absolute_error(y_val, y_pred_et))
print("Validation RMSE:", root_mean_squared_error(y_val, y_pred_et))


Fitting 5 folds for each of 864 candidates, totalling 4320 fits
Best CV RMSE (ExtraTrees): 3.532075044282643
Best params (ExtraTrees):
  model__bootstrap: False
  model__max_depth: None
  model__max_features: 1.0
  model__min_samples_leaf: 1
  model__min_samples_split: 2
  model__n_estimators: 200
Validation R²: 0.8368194400979359
Validation MAE: 2.552375714162424
Validation RMSE: 3.1513911645466335


In [27]:
y_pred_xgb = best_xgb_model.predict(X_test)

print("Test R²:", r2_score(y_test, y_pred_xgb))
print("Test MAE:", mean_absolute_error(y_test, y_pred_xgb))
print("Test RMSE:", root_mean_squared_error(y_test, y_pred_xgb))


Test R²: 0.8083784749446565
Test MAE: 2.1452568348815553
Test RMSE: 3.17903080465149


In [39]:
y_pred_et = best_et_model.predict(X_test)

print("Test R²:", r2_score(y_test, y_pred_et))
print("Test MAE:", mean_absolute_error(y_test, y_pred_et))
print("Test RMSE:", root_mean_squared_error(y_test, y_pred_et))

Test R²: 0.8655570861329972
Test MAE: 1.780653375647875
Test RMSE: 2.6628183948906465


In [30]:
joblib.dump(best_xgb_model, 'models/best_xgb_model.joblib')
joblib.dump(best_et_model, 'models/best_et_model.joblib')

['models/best_et_model.joblib']